# SkyEye — 天气分类
## EfficientNet-B4 → 知识蒸馏 → B0 → ONNX 导出 + INT8 量化

按顺序执行以下 Cell 即可完成完整训练管线。

> 依赖已预装在 `.venv/` 中，激活: `.venv\\Scripts\\activate`

## 1. 环境检查

检查 Python、PyTorch、CUDA 版本及 GPU 显存信息。

In [ ]:
import sys, os
print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

print("\n" + "=" * 60)
print("✓ 环境检查通过")
print("=" * 60)

## 2. 数据准备 + DataLoader

### 2a. 数据准备

自动发现 `datasets/` 下所有数据源，合并到 `_data/weather`。（首次运行复制数据，后续自动跳过）

In [ ]:
from config import CONFIG
from data.dataset import prepare_data

data_root = prepare_data()
print(f"\n{'=' * 60}")
print(f"✓ 数据准备完成 → {data_root}")
print("=" * 60)

### 2b. 加载数据集 + 划分训练/验证集

加载 ImageFolder，统计类别分布，按 85/15 分层划分。

In [ ]:
import numpy as np
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from config import CONFIG

assert 'data_root' in dir(), "请先运行 2a（数据准备）"

full_dataset = ImageFolder(data_root)
class_counts = np.bincount(full_dataset.targets, minlength=len(full_dataset.classes))

indices = np.arange(len(full_dataset))
train_idx, val_idx = train_test_split(
    indices, test_size=CONFIG["val_split"],
    stratify=full_dataset.targets, random_state=CONFIG["seed"],
)

print(f"Classes: {full_dataset.classes}")
print(f"Distribution: {dict(zip(full_dataset.classes, class_counts.astype(int)))}")
print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")
print(f"\n{'=' * 60}")
print("✓ 数据集加载 + 划分完成")
print("=" * 60)

### 2c. 构建 DataLoader

对 train/val 子集分别应用数据增强，构建 DataLoader。

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from config import CONFIG
from data.augmentations import get_train_transforms, get_val_transforms

assert 'data_root' in dir(), "请先运行 2a（数据准备）"
assert 'train_idx' in dir(), "请先运行 2b（数据集划分）"

train_ds = torch.utils.data.Subset(
    ImageFolder(data_root, transform=get_train_transforms(CONFIG["img_size"])), train_idx)
val_ds = torch.utils.data.Subset(
    ImageFolder(data_root, transform=get_val_transforms(CONFIG["img_size"])), val_idx)

nw, pin = CONFIG["num_workers"], torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=nw, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                        num_workers=nw, pin_memory=pin)

print(f"Device: {CONFIG['device']} | Batch: {CONFIG['batch_size']}")
print(f"Workers: {nw} | Pin Memory: {pin}")
print(f"\n{'=' * 60}")
print("✓ DataLoader 构建完成")
print("=" * 60)

## 3. 训练教师模型 (EfficientNet-B4)

分两阶段训练，checkpoint 输出到 `results/checkpoints/teacher/`。具体超参数见 `config.py`。

### 3a. Fast+MU — 标准采样 + MixUp α=0.2 + warmup

In [1]:
from training.train_teacher import train_teacher_phase1
teacher = train_teacher_phase1()
print("\n" + "=" * 60)
print("✓ Phase 1 (Fast+MU) 完成")
print("=" * 60)

Using device: cuda

  Phase 1: Fast+MU — 9 epochs, standard sampling
Cloudy oversampling: OFF (8500 original samples)
Classes: ['cloudy', 'foggy', 'rainy', 'snowy', 'sunny', 'thundery']
Class distribution: {'cloudy': np.int64(10000), 'foggy': np.int64(10000), 'rainy': np.int64(10000), 'snowy': np.int64(10000), 'sunny': np.int64(10000), 'thundery': np.int64(10000)}
Train samples: 51000, Val samples: 9000
TensorBoard 日志目录: results/tb_results/teacher\20260608_230220
Classes: ['cloudy', 'foggy', 'rainy', 'snowy', 'sunny', 'thundery']
Class distribution: {'cloudy': np.int64(10000), 'foggy': np.int64(10000), 'rainy': np.int64(10000), 'snowy': np.int64(10000), 'sunny': np.int64(10000), 'thundery': np.int64(10000)}
Train samples: 51000, Val samples: 9000
LR: 5e-05, MixUp α: 0.2, Warmup: 2 epochs



P1 Epoch 0/8 [Fast+MU]: 100%|██████████| 6375/6375 [13:30<00:00,  7.87it/s, loss=0.4979]


Epoch 0: Train Loss=1.0211 │ Train F1=0.3724 │ Val F1=0.4126 │ Gap=-0.0402 │ Val Acc=45.29%
  ✓ Best saved! F1=0.4126 → fast_mu_best.pth
  ✓ Best saved! F1=0.4126 → teacher_best.pth


P1 Epoch 1/8 [Fast+MU]: 100%|██████████| 6375/6375 [13:17<00:00,  7.99it/s, loss=0.3324]


Epoch 1: Train Loss=0.6358 │ Train F1=0.4918 │ Val F1=0.6709 │ Gap=-0.1790 │ Val Acc=67.44%
  ✓ Best saved! F1=0.6709 → fast_mu_best.pth
  ✓ Best saved! F1=0.6709 → teacher_best.pth


...

Epoch 6: Train Loss=0.3544 │ Train F1=0.5718 │ Val F1=0.8228 │ Gap=-0.2510 │ Val Acc=81.79%
  ✓ Best saved! F1=0.8228 → fast_mu_best.pth
  ✓ Best saved! F1=0.8228 → teacher_best.pth


P1 Epoch 7/8 [Fast+MU]: 100%|██████████| 6375/6375 [13:07<00:00,  8.09it/s, loss=0.1557]


Epoch 7: Train Loss=0.3449 │ Train F1=0.5827 │ Val F1=0.8313 │ Gap=-0.2486 │ Val Acc=82.76%
  ✓ Best saved! F1=0.8313 → fast_mu_best.pth
  ✓ Best saved! F1=0.8313 → teacher_best.pth


P1 Epoch 8/8 [Fast+MU]: 100%|██████████| 6375/6375 [13:05<00:00,  8.12it/s, loss=0.4789]


Epoch 8: Train Loss=0.3350 │ Train F1=0.5822 │ Val F1=0.8448 │ Gap=-0.2626 │ Val Acc=84.21%
  ✓ Best saved! F1=0.8448 → fast_mu_best.pth
  ✓ Best saved! F1=0.8448 → teacher_best.pth

Phase 1 done. Best F1: 0.8448

✓ Phase 1 (Fast+MU) 完成


### 3b. Fast+OS+MU — DRW 过采样 + MixUp α=0.2

In [ ]:
from training.train_teacher import train_teacher_phase2
teacher = train_teacher_phase2()
print("\n" + "=" * 60)
print("✓ Phase 2 (Fast+OS+MU) 完成")
print("=" * 60)

Using device: cuda

  Phase 2: Fast+OS+MU — 6 epochs, DRW oversampling
Cloudy oversampling: OFF (8500 original samples)
Classes: ['cloudy', 'foggy', 'rainy', 'snowy', 'sunny', 'thundery']
Class distribution: {'cloudy': np.int64(10000), 'foggy': np.int64(10000), 'rainy': np.int64(10000), 'snowy': np.int64(10000), 'sunny': np.int64(10000), 'thundery': np.int64(10000)}
Train samples: 51000, Val samples: 9000
Cloudy oversampling: 8500 → 17000 (8500 duplicated, 2×)
Train samples: 59500, Val samples: 9000
Loaded: results/checkpoints/teacher\fast_mu_best.pth
TensorBoard 日志目录: results/tb_results/teacher\20260609_010752
Classes: ['cloudy', 'foggy', 'rainy', 'snowy', 'sunny', 'thundery']
Train samples: 59500, Val samples: 9000
LR: 2.5e-05, MixUp α: 0.2



P2 Epoch 9/14 [Fast+OS+MU]: 100%|██████████| 7438/7438 [15:22<00:00,  8.06it/s, loss=0.0302]


Epoch 9: Train Loss=0.4044 │ Train F1=0.5585 │ Val F1=0.8212 │ Gap=-0.2628 │ Val Acc=81.73%
  ✓ Best saved! F1=0.8212 → fast_os_mu_best.pth
  ✓ Best saved! F1=0.8212 → teacher_best.pth


P2 Epoch 10/14 [Fast+OS+MU]: 100%|██████████| 7438/7438 [15:16<00:00,  8.12it/s, loss=0.3648]


Epoch 10: Train Loss=0.3786 │ Train F1=0.5663 │ Val F1=0.8319 │ Gap=-0.2656 │ Val Acc=82.60%
  ✓ Best saved! F1=0.8319 → fast_os_mu_best.pth
  ✓ Best saved! F1=0.8319 → teacher_best.pth


...

P2 Epoch 12/14 [Fast+OS+MU]: 100%|██████████| 7438/7438 [15:15<00:00,  8.13it/s, loss=0.1642]


Epoch 12: Train Loss=0.3440 │ Train F1=0.5872 │ Val F1=0.8600 │ Gap=-0.2727 │ Val Acc=85.56%
  ✓ Best saved! F1=0.8600 → fast_os_mu_best.pth
  ✓ Best saved! F1=0.8600 → teacher_best.pth


P2 Epoch 13/14 [Fast+OS+MU]: 100%|██████████| 7438/7438 [15:21<00:00,  8.07it/s, loss=0.6053]


Epoch 13: Train Loss=0.3358 │ Train F1=0.5859 │ Val F1=0.8618 │ Gap=-0.2758 │ Val Acc=85.81%
  ✓ Best saved! F1=0.8618 → fast_os_mu_best.pth
  ✓ Best saved! F1=0.8618 → teacher_best.pth


P2 Epoch 14/14 [Fast+OS+MU]:  41%|████▏     | 3077/7438 [06:33<09:07,  7.96it/s, loss=0.0651]

## 4. 知识蒸馏 (B4 → B0)

用教师模型（B4）的知识蒸馏训练学生模型（B0），输出 `results/student_distilled_best.pth`。

In [ ]:
from training.distill_student import run_distillation
student = run_distillation()
print("\n" + "=" * 60)
print("✓ 知识蒸馏完成")
print("=" * 60)

## 5. ONNX 导出 + INT8 量化 + CPU 测速

将蒸馏后的 B0 学生模型导出，量化，并验证推理速度。

### 5a. ONNX 导出

输出 `results/weather_model.onnx`

In [ ]:
from inference.export_onnx import export_to_onnx
onnx_path = export_to_onnx('results/student_distilled_best.pth')
print("\n" + "=" * 60)
print("✓ ONNX 导出完成")
print("=" * 60)

### 5b. INT8 动态量化

输出 `results/weather_model_int8.onnx`

In [ ]:
from inference.export_onnx import quantize_to_int8
int8_path = quantize_to_int8(onnx_path)
print("\n" + "=" * 60)
print("✓ INT8 量化完成")
print("=" * 60)

### 5c. CPU 推理测速

In [ ]:
from inference.export_onnx import benchmark_cpu
benchmark_cpu(onnx_path, int8_path)
print("\n" + "=" * 60)
print("✓ CPU 测速完成")
print("=" * 60)

## 6. (可选) CPU 单张推理测试

用导出的模型对单张图片进行推理验证。

In [ ]:
from inference.infer import predict_image
result = predict_image('your_image.jpg')
print(f'Prediction: {result["prediction"]} (Confidence: {result["confidence"]:.4f})')
for name, prob in result['top_k']:
    print(f'  {name}: {prob:.4f}')